# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR\^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset is defined by a Croissant schema available from:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

All dataset entities such as record sets, fields, and columns are referenced using their `@id`, consistent with the Croissant specification.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset and its metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Print dataset metadata (accessing known properties, e.g., .name, .description)
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview

List available record sets and their fields with corresponding `@id`.

For each record set, we'll display its `@id` and title, and for each of its fields, its `@id`, name, and data type.

In [ ]:
from pprint import pprint

# List all record sets
print("Available Record Sets (by @id):")
record_set_ids = []
for record_set in dataset.metadata.record_sets:
    print(f"  - @id: {record_set.id}")
    print(f"    name: {record_set.name if hasattr(record_set, 'name') else ''}")
    record_set_ids.append(record_set.id)

    # List fields for this record set
    if hasattr(record_set, 'fields'):
        print("    Fields:")
        for field in record_set.fields:
            print(f"      - @id: {field.id}")
            print(f"        name: {field.name if hasattr(field, 'name') else ''}")
            print(f"        dataType: {field.data_type if hasattr(field, 'data_type') else ''}")
    print()

## 3. Data Extraction

Load data from all available record sets into Pandas DataFrames. Use the record set and field `@id`s from the overview above.

In [ ]:
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Extracting records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"No records found for {record_set_id}.")


## 4. Exploratory Data Analysis (EDA)

Demonstrate example EDA operations on a key data table from the dataset:
- Filtering records based on a numeric field (e.g., by molecular weight or abundance)
- Normalizing this field
- Grouping by an annotation/category field

> **Note:** You need to replace `<record_set_id>`, `<numeric_field>`, or `<group_field>` with actual `@id` names visible in the output above, depending on your analysis.

In [ ]:
# Select the primary record set. Replace this string with the @id of your main table after inspecting previous cell outputs.
main_record_set_id = record_set_ids[0] if record_set_ids else None  # Use the first available one as an example

if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Guess a numeric field (e.g., one containing "MW", "Abundance", or "Count")
    numeric_cols = df.select_dtypes(include='number').columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notna().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}")
        display(filtered_df.head())

        # Normalize
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized values for {numeric_field}:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Try to select a group field (categorical)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == 'object' and df[col].nunique() < 10:
                group_field = col
                break
        if group_field is not None:
            print(f"Grouping by {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No main record set dataframe available for EDA.")

## 5. Visualization

Visualize the distribution of a selected numeric field. For molecular/protein datasets, this may include a histogram of molecular weight, abundance, or peptide count. Replace `main_record_set_id` and `numeric_field` with your choices if needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel("Frequency")
        plt.show()
    else:
        print("No numeric field to visualize.")
else:
    print("No dataframe to visualize.")

## 6. Conclusion

We successfully loaded the mass spectrometry dataset using `mlcroissant`, explored available record sets and fields by their `@id`, and demonstrated common EDA and visualization workflows. Key insights can be extended for more specialized proteomics statistics or FAIRness audits leveraging the complete schema provided by the Croissant package.

> **Next Steps:**
> Further investigation can segment the data by more specific fields, enrich analysis with cross-references in the Croissant metadata, or operationalize the data for machine learning workflows.